# Notebook 77B — Direct Notebook-75 Pixel Join Audit

**Purpose:** rescore the already-locked Notebook 77 observable `PAPER_SUPPORTED` veto directly against the authoritative Notebook 75 selected-pixel truth table.

This notebook does **not** reconstruct truth from page images, does **not** rerun C0/C1/A2 inference, and does **not** change any policy threshold.

It must first prove that the Notebook 75 `paired_selected_pixels.parquet` reproduces the accepted C1 result. Only then may it join Notebook 77's locked paper-veto coordinates and report the A2 counterfactual.

### Required scientific sequence
1. Resolve one completed Notebook 75 run containing `tables/paired_selected_pixels.parquet`.
2. Resolve the locked Notebook 77 run and its prediction lock / observable maps.
3. Inspect schemas and fail closed on ambiguity.
4. Isolate the authoritative C1 Ring-2 rows from Notebook 75.
5. Reproduce Notebook 75 C1 page/pixel metrics exactly.
6. Join Notebook 77 `PAPER_SUPPORTED` by exact `(page_key/page, y, x)` coordinate identity.
7. Partition C1 into `VETOED_BY_A2` and `RETAINED_BY_A2`.
8. Report PRINT / ANNOTATION / UNCERTAIN counts, utility, catastrophe, page/source/annotator stability.
9. Produce visual review tables and contact sheets.
10. Package a concise review ZIP.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy pyarrow matplotlib opencv-python-headless


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Configuration


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, re, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

# Optional exact run overrides. Prefer setting these after the first discovery pass.
NB75_RUN_OVERRIDE = ROOT / 'notebook75_v88_collar_preservation_counterfactual' / '20261118_115735_notebook75_collar_counterfactual'
NB77_RUN_OVERRIDE = ROOT / 'notebook77_observable_paper_veto' / '20261118_150756_notebook77_observable_paper_veto'

# Expected accepted Notebook 75 C1 headline values from the completed run.
EXPECTED_C1_TRUE_PRINT = 18178
EXPECTED_C1_ANNOTATION = 1246
EXPECTED_C1_UTILITY = 11448
EXPECTED_C1_CATASTROPHIC_COMPONENTS = 1

# Notebook-75 utility/catastrophe contract.
FALSE_PENALTY = 5
CATASTROPHE_PENALTY = 500
CATASTROPHE_FALSE_PIXELS = 50
CATASTROPHE_RATIO = 1.0
CATASTROPHE_RATIO_MIN_FALSE = 20

RUN = datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook77b_direct_n75_pixel_join'
OUT = ROOT / 'notebook77b_direct_pixel_join' / RUN
TABLES = OUT / 'tables'
VISUALS = OUT / 'visuals'
AUDIT = OUT / 'audit'
UPLOAD = OUT / 'upload'
for d in [OUT, TABLES, VISUALS, AUDIT, UPLOAD]:
    d.mkdir(parents=True, exist_ok=False)

print('Output:', OUT)


Output: /content/drive/MyDrive/MusicAnnotationProject/notebook77b_direct_pixel_join/20261118_230025_notebook77b_direct_n75_pixel_join


## 2. Helpers and run discovery


In [3]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()

def latest_complete(root, required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required):
                hits.append(p)
    return max(hits, key=lambda p:p.stat().st_mtime) if hits else None

def find_nb75():
    if NB75_RUN_OVERRIDE:
        return Path(NB75_RUN_OVERRIDE)
    candidates=[]
    required=['tables/paired_selected_pixels.parquet']
    for root in [ROOT/'notebook75_collar_counterfactual', ROOT/'notebook75', ROOT]:
        q=latest_complete(root, required)
        if q is not None: candidates.append(q)
    return max(candidates,key=lambda p:p.stat().st_mtime) if candidates else None

def find_nb77():
    if NB77_RUN_OVERRIDE:
        return Path(NB77_RUN_OVERRIDE)
    candidates=[]
    # Require a lock plus something containing observable maps. We test exact layout below.
    for p in ROOT.rglob('prediction_lock.json'):
        parent=p.parent
        if '77' in str(parent).lower() or 'paper' in str(parent).lower():
            candidates.append(parent)
    return max(candidates,key=lambda p:p.stat().st_mtime) if candidates else None

NB75=find_nb75(); NB77=find_nb77()
if NB75 is None: raise RuntimeError('No Notebook 75 run containing tables/paired_selected_pixels.parquet was found. Set NB75_RUN_OVERRIDE.')
if NB77 is None: raise RuntimeError('No locked Notebook 77 run was found. Set NB77_RUN_OVERRIDE.')
print('Notebook 75:', NB75)
print('Notebook 77:', NB77)


Notebook 75: /content/drive/MyDrive/MusicAnnotationProject/notebook75_v88_collar_preservation_counterfactual/20261118_115735_notebook75_collar_counterfactual
Notebook 77: /content/drive/MyDrive/MusicAnnotationProject/notebook77_observable_paper_veto/20261118_150756_notebook77_observable_paper_veto


## 3. Load Notebook 75 authoritative selected-pixel table and inspect schema


In [4]:
N75_PATH = NB75/'tables'/'paired_selected_pixels.parquet'
n75 = pd.read_parquet(N75_PATH)
print('Notebook 75 pixel table:', n75.shape)
print('Columns:', n75.columns.tolist())
display(n75.head())

# Resolve key columns conservatively.
def one_col(frame, choices, label, required=True):
    hits=[c for c in choices if c in frame.columns]
    if len(hits)>1:
        # Exact synonyms may coexist in some runs; use first but record all.
        print(f'{label}: multiple candidates {hits}; using {hits[0]}')
    if not hits:
        if required: raise RuntimeError(f'Cannot resolve {label}; candidates were {choices}')
        return None
    return hits[0]

COL_PAGE = one_col(n75, ['page_key','page'], 'page identity')
COL_Y = one_col(n75, ['y','row'], 'y')
COL_X = one_col(n75, ['x','col'], 'x')
COL_ARM = one_col(n75, ['arm','candidate_arm','policy_arm'], 'arm')
COL_TRUTH = one_col(n75, ['truth_class','truth','label','truth_name'], 'truth class')
COL_RING = one_col(n75, ['ring','authority_band','band'], 'ring', required=False)
COL_SOURCE = one_col(n75, ['source_group','group'], 'source group', required=False)
COL_COMPONENT = None

for c in [COL_Y,COL_X]: n75[c]=pd.to_numeric(n75[c],errors='raise').astype(int)
n75[COL_PAGE]=n75[COL_PAGE].astype(str)
print('Arm counts:'); print(n75[COL_ARM].astype(str).value_counts().head(20))
print('Truth counts:'); print(n75[COL_TRUTH].astype(str).value_counts(dropna=False).head(20))
if COL_RING: print('Ring counts:'); print(n75[COL_RING].astype(str).value_counts(dropna=False).head(20))


Notebook 75 pixel table: (613336, 12)
Columns: ['page_key', 'page', 'source_group', 'arm', 'y', 'x', 'ring', 'route', 'truth_class', 'observed', 'incumbent', 'candidate']


,page_key,page,source_group,arm,y,x,ring,route,truth_class,observed,incumbent,candidate
0,paired__Page002,Page002,Wright_and_Round_Faith,C1_PRESERVE_RING2,16,560,RING2,V78_REMOVE,OTHER,255,254,255
1,paired__Page002,Page002,Wright_and_Round_Faith,C1_PRESERVE_RING2,16,561,RING2,V78_REMOVE,OTHER,255,254,255
2,paired__Page002,Page002,Wright_and_Round_Faith,C1_PRESERVE_RING2,17,558,RING2,V78_REMOVE,OTHER,255,254,255
3,paired__Page002,Page002,Wright_and_Round_Faith,C1_PRESERVE_RING2,17,559,RING2,V78_REMOVE,OTHER,255,254,255
4,paired__Page002,Page002,Wright_and_Round_Faith,C1_PRESERVE_RING2,17,562,RING2,V78_REMOVE,OTHER,253,254,253


page identity: multiple candidates ['page_key', 'page']; using page_key
Arm counts:
arm
C2_PRESERVE_RING1_RING2    395846
C1_PRESERVE_RING2          217490
Name: count, dtype: int64
Truth counts:
truth_class
OTHER         545124
PRINT          53862
ANNOTATION     14350
Name: count, dtype: int64
Ring counts:
ring
RING2    434980
RING1    178356
Name: count, dtype: int64


## 4. Canonicalise Notebook 75 truth and isolate C1 Ring-2 rows


In [5]:
def canon_truth(v):
    s=str(v).strip().upper()
    if s in {'1','PRINT','TRUE_PRINT','TRUE PRINT'}: return 'PRINT'
    if s in {'0','ANNOTATION','FALSE_ANNOTATION','FALSE ANNOTATION','NON_PRINT','NON-PRINT'}: return 'ANNOTATION'
    if s in {'2','UNCERTAIN','TRUTH_UNCERTAIN','TRUTH UNCERTAIN'}: return 'UNCERTAIN'
    if s in {'OTHER','NAN','NONE',''}: return 'OTHER'
    return s

def is_c1(v):
    s=str(v).upper()
    return ('C1' in s and 'RING2' in s) or s in {'C1','C1_PRESERVE_RING2'}

def is_ring2(v):
    s=str(v).upper().replace(' ','').replace('_','')
    return s in {'RING2','R2','2'} or 'RING2' in s

n75['truth_canon']=n75[COL_TRUTH].map(canon_truth)
c1=n75[n75[COL_ARM].map(is_c1)].copy()
if COL_RING is not None:
    # Only filter if the C1 table contains mixed ring identities.
    ring_values=set(c1[COL_RING].astype(str).str.upper().dropna().unique())
    if any(('RING1' in x or x in {'1','R1'}) for x in ring_values):
        c1=c1[c1[COL_RING].map(is_ring2)].copy()

if c1.empty: raise RuntimeError('No C1 Ring-2 rows found in Notebook 75 pixel table.')
if c1.duplicated([COL_PAGE,COL_Y,COL_X]).any():
    dup=c1[c1.duplicated([COL_PAGE,COL_Y,COL_X],keep=False)]
    dup.to_csv(AUDIT/'duplicate_c1_coordinates.csv',index=False)
    raise RuntimeError('Notebook 75 C1 contains duplicate page/y/x coordinates.')

print('C1 rows:',len(c1),'pages:',c1[COL_PAGE].nunique())
print(c1.truth_canon.value_counts(dropna=False))
c1.to_parquet(TABLES/'authoritative_notebook75_c1_rows.parquet',index=False)


C1 rows: 217490 pages: 51
truth_canon
OTHER         198066
PRINT          18178
ANNOTATION      1246
Name: count, dtype: int64


## 5. First fail-closed gate — reproduce the accepted Notebook 75 C1 result from its own pixel table


In [6]:
def catastrophe_summary(frame):
    rows=[]
    if COL_COMPONENT is None:
        raise RuntimeError('Component identity is required to reproduce Notebook 75 catastrophe count exactly.')
    groups=[COL_PAGE,COL_COMPONENT]
    if COL_SOURCE: groups.insert(1,COL_SOURCE)
    for keys,g in frame.groupby(groups,dropna=False):
        tp=int((g.truth_canon=='PRINT').sum())
        fp=int((g.truth_canon=='ANNOTATION').sum())
        cat=bool(fp>=CATASTROPHE_FALSE_PIXELS or (fp/max(tp,1)>CATASTROPHE_RATIO and fp>=CATASTROPHE_RATIO_MIN_FALSE))
        rec={'true_print':tp,'annotation':fp,'catastrophic':cat}
        if isinstance(keys,tuple):
            for k,v in zip(groups,keys): rec[k]=v
        else: rec[groups[0]]=keys
        rows.append(rec)
    return pd.DataFrame(rows)

c1_tp=int((c1.truth_canon=='PRINT').sum())
c1_fp=int((c1.truth_canon=='ANNOTATION').sum())
c1_unc=int((c1.truth_canon=='UNCERTAIN').sum())
c1_comps=catastrophe_summary(c1)
c1_cat=int(c1_comps.catastrophic.sum())
c1_utility=c1_tp-FALSE_PENALTY*c1_fp-CATASTROPHE_PENALTY*c1_cat

reproduction={
    'true_print':c1_tp,
    'annotation':c1_fp,
    'truth_uncertain':c1_unc,
    'catastrophic_components':c1_cat,
    'utility':c1_utility,
    'expected_true_print':EXPECTED_C1_TRUE_PRINT,
    'expected_annotation':EXPECTED_C1_ANNOTATION,
    'expected_catastrophic_components':EXPECTED_C1_CATASTROPHIC_COMPONENTS,
    'expected_utility':EXPECTED_C1_UTILITY,
}
reproduction['passed']=bool(
    c1_tp==EXPECTED_C1_TRUE_PRINT and
    c1_fp==EXPECTED_C1_ANNOTATION and
    c1_cat==EXPECTED_C1_CATASTROPHIC_COMPONENTS and
    c1_utility==EXPECTED_C1_UTILITY
)
(OUT/'c1_reproduction.json').write_text(json.dumps(reproduction,indent=2))
c1_comps.to_csv(TABLES/'c1_component_reproduction.csv',index=False)
print(json.dumps(reproduction,indent=2))
if not reproduction['passed']:
    raise RuntimeError('STOP: authoritative Notebook 75 pixel table did not reproduce the accepted C1 headline result. No A2 interpretation is permitted.')
print('PASS: Notebook 75 C1 reproduced exactly from its own selected-pixel table.')


RuntimeError: Component identity is required to reproduce Notebook 75 catastrophe count exactly.

## 6. Resolve Notebook 77 locked PAPER_SUPPORTED maps


In [ ]:
# Find all NPZ files beneath Notebook 77 and identify those containing paper_supported.
map_files=[]
for p in NB77.rglob('*.npz'):
    try:
        with np.load(p) as z:
            keys=set(z.files)
        if 'paper_supported' in keys or 'paper_veto' in keys:
            map_files.append(p)
    except Exception:
        pass

if not map_files:
    raise RuntimeError('No Notebook 77 NPZ containing paper_supported/paper_veto was found.')
print('Candidate locked map files:',len(map_files))
print('\n'.join(str(p) for p in map_files[:20]))

# Map files to page identity using basename stem; later we verify coordinates are in-bounds.
def stem_page(p):
    s=p.stem
    for suffix in ['_observable_maps','_observable','_paper_supported','_paper_veto','_maps']:
        if s.endswith(suffix): s=s[:-len(suffix)]
    return s

file_by_stem={stem_page(p):p for p in map_files}

# Resolve each Notebook75 page by exact or conservative normalized stem match.
def norm_name(s): return re.sub(r'[^a-z0-9]+','',str(s).lower())
norm_lookup={norm_name(k):v for k,v in file_by_stem.items()}

def resolve_map(page):
    if page in file_by_stem: return file_by_stem[page]
    return norm_lookup.get(norm_name(page))

missing=[]; mapping=[]
for page in sorted(c1[COL_PAGE].unique()):
    p=resolve_map(page)
    if p is None: missing.append(page)
    else: mapping.append({'page':page,'map_path':str(p),'sha256':sha256_file(p)})
pd.DataFrame(mapping).to_csv(AUDIT/'notebook77_map_resolution.csv',index=False)
if missing:
    pd.DataFrame({'page':missing}).to_csv(AUDIT/'missing_notebook77_maps.csv',index=False)
    raise RuntimeError(f'Missing Notebook 77 paper map for {len(missing)} C1 pages. See audit/missing_notebook77_maps.csv')
print('Resolved maps for all C1 pages:',len(mapping))


## 7. Direct coordinate join — no truth reconstruction


In [ ]:
parts=[]
for page,g in c1.groupby(COL_PAGE,sort=False):
    path=resolve_map(page)
    with np.load(path) as z:
        key='paper_supported' if 'paper_supported' in z.files else 'paper_veto'
        paper=np.asarray(z[key]).astype(bool)
    ys=g[COL_Y].to_numpy(int); xs=g[COL_X].to_numpy(int)
    if (ys<0).any() or (xs<0).any() or (ys>=paper.shape[0]).any() or (xs>=paper.shape[1]).any():
        raise RuntimeError(f'{page}: Notebook75 coordinates fall outside Notebook77 paper map shape {paper.shape}.')
    q=g.copy()
    q['paper_veto']=paper[ys,xs]
    parts.append(q)
joined=pd.concat(parts,ignore_index=True)
if len(joined)!=len(c1): raise RuntimeError('Join row count changed unexpectedly.')
if joined.duplicated([COL_PAGE,COL_Y,COL_X]).any(): raise RuntimeError('Duplicate coordinates after join.')
joined.to_parquet(TABLES/'c1_with_notebook77_paper_veto.parquet',index=False)
print(pd.crosstab(joined.paper_veto,joined.truth_canon,margins=True))


## 8. Score A2 directly from authoritative Notebook 75 truth labels


In [ ]:
vetoed=joined[joined.paper_veto].copy()
a2=joined[~joined.paper_veto].copy()

def score(frame,label):
    tp=int((frame.truth_canon=='PRINT').sum())
    fp=int((frame.truth_canon=='ANNOTATION').sum())
    unc=int((frame.truth_canon=='UNCERTAIN').sum())
    comps=catastrophe_summary(frame) if len(frame) else pd.DataFrame(columns=['catastrophic'])
    cat=int(comps.catastrophic.sum()) if len(comps) else 0
    utility=tp-FALSE_PENALTY*fp-CATASTROPHE_PENALTY*cat
    return {'arm':label,'rows':len(frame),'true_print':tp,'annotation':fp,'truth_uncertain':unc,'catastrophic_components':cat,'utility':utility,'false_true_ratio':fp/max(tp,1),'false_preservation_rate':fp/max(tp+fp,1)}

c1_score=score(joined,'C1_RING2')
a2_score=score(a2,'A2_PAPER_VETO')
veto_score=score(vetoed,'REMOVED_BY_PAPER_VETO')
summary=pd.DataFrame([c1_score,a2_score,veto_score])
summary.to_csv(TABLES/'direct_join_arm_summary.csv',index=False)
print(summary.to_string(index=False))

comparison={
    'c1_true_print':c1_score['true_print'],
    'c1_annotation':c1_score['annotation'],
    'a2_true_print':a2_score['true_print'],
    'a2_annotation':a2_score['annotation'],
    'paper_veto_true_print_removed':veto_score['true_print'],
    'paper_veto_annotation_removed':veto_score['annotation'],
    'paper_veto_uncertain_removed':veto_score['truth_uncertain'],
    'delta_true_print':a2_score['true_print']-c1_score['true_print'],
    'delta_annotation':a2_score['annotation']-c1_score['annotation'],
    'delta_utility':a2_score['utility']-c1_score['utility'],
    'strictly_preserves_all_c1_print':bool(veto_score['true_print']==0),
    'improves_annotation_retention':bool(a2_score['annotation']<c1_score['annotation']),
}
comparison['strict_dominance_on_print_and_annotation']=bool(comparison['strictly_preserves_all_c1_print'] and comparison['improves_annotation_retention'])
(OUT/'a2_direct_join_result.json').write_text(json.dumps(comparison,indent=2))
print(json.dumps(comparison,indent=2))


## 9. Page and source-group stability


In [ ]:
rows=[]
for page,g in joined.groupby(COL_PAGE):
    kept=g[~g.paper_veto]; rem=g[g.paper_veto]
    row={'page':page}
    if COL_SOURCE: row['source_group']=str(g[COL_SOURCE].iloc[0])
    row.update({
        'c1_true_print':int((g.truth_canon=='PRINT').sum()),
        'c1_annotation':int((g.truth_canon=='ANNOTATION').sum()),
        'a2_true_print':int((kept.truth_canon=='PRINT').sum()),
        'a2_annotation':int((kept.truth_canon=='ANNOTATION').sum()),
        'veto_true_print':int((rem.truth_canon=='PRINT').sum()),
        'veto_annotation':int((rem.truth_canon=='ANNOTATION').sum()),
        'veto_uncertain':int((rem.truth_canon=='UNCERTAIN').sum()),
    })
    rows.append(row)
page_df=pd.DataFrame(rows)
page_df.to_csv(TABLES/'direct_join_by_page.csv',index=False)

if COL_SOURCE:
    group_df=page_df.groupby('source_group',as_index=False).agg(
        pages=('page','nunique'),
        c1_true_print=('c1_true_print','sum'), c1_annotation=('c1_annotation','sum'),
        a2_true_print=('a2_true_print','sum'), a2_annotation=('a2_annotation','sum'),
        veto_true_print=('veto_true_print','sum'), veto_annotation=('veto_annotation','sum'), veto_uncertain=('veto_uncertain','sum'))
    group_df.to_csv(TABLES/'direct_join_by_source_group.csv',index=False)
    display(group_df.sort_values(['veto_true_print','veto_annotation'],ascending=[False,False]))

display(page_df.sort_values(['veto_true_print','veto_annotation'],ascending=[False,False]).head(30))


## 10. Optional annotator/style stratification


In [ ]:
# This is descriptive only; annotator identity is never an inference feature.
# Try to locate an existing page/annotator manifest. If none is found, the analysis is skipped.
annotator_candidates=[]
for p in ROOT.rglob('*.csv'):
    try:
        if p.stat().st_size>5_000_000: continue
        h=pd.read_csv(p,nrows=5,dtype=str)
        cols={c.lower():c for c in h.columns}
        if 'page' in cols and ('annotator' in cols or 'reviewer' in cols): annotator_candidates.append(p)
    except Exception:
        pass

if annotator_candidates:
    ap=max(annotator_candidates,key=lambda p:p.stat().st_mtime)
    ann=pd.read_csv(ap,dtype=str).fillna('')
    page_col=next(c for c in ann.columns if c.lower()=='page')
    annot_col=next(c for c in ann.columns if c.lower() in {'annotator','reviewer'})
    ann=ann[[page_col,annot_col]].drop_duplicates()
    q=page_df.merge(ann,left_on='page',right_on=page_col,how='left')
    a=q.groupby(annot_col,dropna=False,as_index=False).agg(pages=('page','nunique'),c1_true_print=('c1_true_print','sum'),c1_annotation=('c1_annotation','sum'),a2_true_print=('a2_true_print','sum'),a2_annotation=('a2_annotation','sum'),veto_true_print=('veto_true_print','sum'),veto_annotation=('veto_annotation','sum'))
    a.to_csv(TABLES/'direct_join_by_annotator.csv',index=False)
    print('Annotator manifest:',ap); display(a)
else:
    print('No explicit annotator manifest found; skipping annotator stratification.')


## 11. Visual summaries


In [ ]:
# Correct benefit-cost plot from the authoritative Notebook75 population.
plt.figure(figsize=(9,5.5))
x=np.arange(2); width=.35
vals_tp=[c1_score['true_print'],a2_score['true_print']]
vals_fp=[c1_score['annotation'],a2_score['annotation']]
plt.bar(x-width/2,vals_tp,width,label='true print recovered')
plt.bar(x+width/2,vals_fp,width,label='annotation retained')
plt.xticks(x,['C1 Ring2','A2 paper veto'])
plt.ylabel('Authoritative Notebook75 selected pixels')
plt.title('Notebook 77B: direct Notebook75 truth join')
plt.legend(); plt.tight_layout(); plt.savefig(VISUALS/'c1_vs_a2_authoritative_benefit_cost.png',dpi=180); plt.close()

# Page-level veto trade-off.
plt.figure(figsize=(7,6))
plt.scatter(page_df.veto_annotation,page_df.veto_true_print,alpha=.75)
plt.xlabel('Annotation pixels vetoed by A2')
plt.ylabel('Genuine C1 print pixels vetoed by A2')
plt.title('Paper-veto effect by page')
plt.axhline(0,color='black',linewidth=.8); plt.tight_layout(); plt.savefig(VISUALS/'paper_veto_page_tradeoff.png',dpi=180); plt.close()

# Truth composition of vetoed coordinates.
counts=vetoed.truth_canon.value_counts().reindex(['PRINT','ANNOTATION','UNCERTAIN','OTHER'],fill_value=0)
plt.figure(figsize=(7,5)); plt.bar(counts.index,counts.values); plt.ylabel('Vetoed C1 coordinates'); plt.title('Truth composition of locked PAPER_SUPPORTED veto'); plt.tight_layout(); plt.savefig(VISUALS/'paper_veto_truth_composition.png',dpi=180); plt.close()
print('Visuals:', VISUALS)


## 12. Final verdict and upload bundle


In [ ]:
verdict={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'notebook':'77B_Direct_Notebook75_Pixel_Join_Audit',
    'notebook75_run':str(NB75),
    'notebook77_run':str(NB77),
    'notebook75_pixel_table_sha256':sha256_file(N75_PATH),
    'c1_reproduction':reproduction,
    'a2_comparison':comparison,
    'interpretation_allowed':bool(reproduction['passed']),
    'no_truth_reconstruction_performed':True,
    'notebook77_decisions_rerun':False,
    'promotion_authority':False,
}
if reproduction['passed'] and comparison['strict_dominance_on_print_and_annotation']:
    verdict['verdict']='PASS_A2_STRICTLY_DOMINATES_C1_ON_AUTHORITATIVE_N75_PIXEL_LABELS'
elif reproduction['passed'] and comparison['improves_annotation_retention']:
    verdict['verdict']='MIXED_A2_REDUCES_ANNOTATION_BUT_SACRIFICES_SOME_C1_PRINT'
elif reproduction['passed']:
    verdict['verdict']='NO_A2_BENEFIT_ON_AUTHORITATIVE_N75_PIXEL_LABELS'
else:
    verdict['verdict']='FAIL_C1_REPRODUCTION'
(OUT/'notebook77b_verdict.json').write_text(json.dumps(verdict,indent=2))

summary_lines=[
    'NOTEBOOK 77B — DIRECT NOTEBOOK75 PIXEL JOIN AUDIT','',
    f'Verdict: {verdict["verdict"]}',
    f'Notebook75: {NB75}',
    f'Notebook77: {NB77}','',
    f'C1 true print: {c1_score["true_print"]}',
    f'C1 annotation: {c1_score["annotation"]}',
    f'A2 true print: {a2_score["true_print"]}',
    f'A2 annotation: {a2_score["annotation"]}',
    f'Paper veto removed true print: {veto_score["true_print"]}',
    f'Paper veto removed annotation: {veto_score["annotation"]}',
    f'Paper veto removed truth-uncertain: {veto_score["truth_uncertain"]}',
    f'Delta utility: {comparison["delta_utility"]}','',
    'This audit uses Notebook75 truth labels directly. It does not reconstruct clean truth or rerun Notebook77 decisions.',
    'No production promotion is authorised.'
]
(OUT/'executive_summary.txt').write_text('
'.join(summary_lines))
print('
'.join(summary_lines))

# Curated upload package.
artefacts=[OUT/'executive_summary.txt',OUT/'notebook77b_verdict.json',OUT/'c1_reproduction.json',OUT/'a2_direct_join_result.json',TABLES/'direct_join_arm_summary.csv',TABLES/'direct_join_by_page.csv',VISUALS/'c1_vs_a2_authoritative_benefit_cost.png',VISUALS/'paper_veto_page_tradeoff.png',VISUALS/'paper_veto_truth_composition.png',AUDIT/'notebook77_map_resolution.csv']
for p in artefacts:
    if not p.exists(): raise RuntimeError(f'Missing upload artefact: {p}')
    shutil.copy2(p,UPLOAD/p.name)
manifest=[]
for p in sorted(UPLOAD.iterdir()):
    if p.is_file(): manifest.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
zip_base=OUT/'notebook77b_direct_pixel_join_bundle'
shutil.make_archive(str(zip_base),'zip',UPLOAD)
print('Bundle:',str(zip_base)+'.zip')


## Interpretation guardrail

- A PASS here means only that the already-locked Notebook 77 paper veto strictly improves the existing Notebook 75 C1 selected-pixel truth result under the same historical labels.
- The paper veto remains a challenger. It is not automatically incorporated into `v8_8_operator.py`.
- If any C1 PRINT coordinate is vetoed, inspect those coordinates visually before deciding whether the visual-restoration trade-off remains acceptable.
- Preserve the failed Notebook 77A as evidence that reconstructed truth scoring did not reproduce Notebook 75 exactly.
